# NAIP-CHM AOI Inference for CSDV PoC Sites

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bullocke/CSDV_RS/blob/main/ProofOfConcept/Code/NAIP_CHM/colab/naip_chm_aoi_inference.ipynb)

Runs the pre-trained NAIP-CHM model (Morford et al. 2025) on a small AOI around a NEON PoC site, for one or more NAIP years. Designed to produce a first usable CHM time series in one Colab session.

Differences from the upstream `gee_inference_colab.ipynb`:
- AOI-clipped instead of full NAIP DOQQ (faster, smaller files).
- Loops over all requested NAIP years for the site.
- Output filenames carry `YYYYMMDD` (median acquisition date of contributing NAIP images), which the upstream `extract_doy_from_filename` requires.
- Calls the standard `scripts/inference.py` CLI rather than the full-DOQQ streamer.

### Runtime
Set Runtime > Change runtime type > T4 GPU before running.

## 1. Config

In [ ]:
# === USER CONFIG ===
SITE = 'SCBI'
LAT, LON = 38.8929, -78.1454   # center of AOI in WGS84 decimal degrees
HALF_SIZE_KM = 2.5             # AOI half-edge; full AOI is 2*HALF_SIZE_KM square

# Years to predict. Set to None to auto-pick all available 4-band NAIP years.
YEARS = [2018, 2021, 2023]

PROJECT_ID = 'dyce-biomass'    # your Earth Engine project
DRIVE_FOLDER = 'NAIP_CHM_PoC'  # output folder under MyDrive

# Upstream repo locations (do not change)
MODEL_PATH = 'model/model_20251016.pt'
CONFIG_PATH = 'configs/config.yaml'
CONDITIONING_DIR = 'data/conditioning_data'

## 2. Clone repo and install dependencies

In [ ]:
%%bash
set -e
if [ ! -d naip-chm ]; then
  git clone --depth 1 https://github.com/smorf-ntsg/naip-chm.git
fi
cd naip-chm
pip install -q -r requirements.txt rio-cogeo geemap

In [ ]:
import os, sys
from pathlib import Path
REPO = Path('/content/naip-chm').resolve()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Working in', REPO)

## 3. Download conditioning rasters (one-time, ~few GB)

Cached to Drive so subsequent sessions skip the download.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive') / DRIVE_FOLDER
DRIVE_COND = DRIVE_ROOT / 'conditioning_data'
DRIVE_NAIP = DRIVE_ROOT / 'naip_aoi'
DRIVE_CHM = DRIVE_ROOT / 'chm_predictions' / SITE
for p in (DRIVE_COND, DRIVE_NAIP, DRIVE_CHM):
    p.mkdir(parents=True, exist_ok=True)

# Symlink local conditioning_data -> Drive so we only download once.
local_cond = REPO / CONDITIONING_DIR
local_cond.parent.mkdir(parents=True, exist_ok=True)
if local_cond.exists() or local_cond.is_symlink():
    if local_cond.is_symlink():
        local_cond.unlink()
    else:
        import shutil; shutil.rmtree(local_cond)
local_cond.symlink_to(DRIVE_COND)
print('Conditioning dir ->', local_cond, '->', DRIVE_COND)

In [ ]:
# Run upstream downloader. Answer 'n' to the sample NAIP download prompt (we use our own).
!echo n | python scripts/download_conditioning_data.py

required = ['elevation.tif', 'climate_pca.tif', 'soil_pca.tif', 'nlcd.tif', 'ecoregion.tif']
missing = [f for f in required if not (local_cond / f).exists()]
assert not missing, f'Missing conditioning rasters: {missing}'
print('All 5 conditioning rasters present.')

## 4. Earth Engine auth and AOI

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project=PROJECT_ID, opt_url='https://earthengine-highvolume.googleapis.com')

# Build AOI in WGS84. We convert to a meter-accurate square by buffering a point.
half_m = HALF_SIZE_KM * 1000.0
center = ee.Geometry.Point([LON, LAT])
aoi = center.buffer(half_m).bounds()  # rectangular bounds of the buffered circle
print('AOI bounds (WGS84):', aoi.bounds().getInfo()['coordinates'])

## 5. List available 4-band NAIP years for this AOI

In [ ]:
naip_col = ee.ImageCollection('USDA/NAIP/DOQQ').filterBounds(aoi)

def _tag(img):
    return img.set({
        'band_count': img.bandNames().length(),
        'year': img.date().get('year'),
        'acq_ms': img.date().millis(),
    })

tagged = naip_col.map(_tag).filter(ee.Filter.eq('band_count', 4))
available_years = sorted(set(tagged.aggregate_array('year').getInfo()))
print('Available 4-band NAIP years for AOI:', available_years)

if YEARS is None:
    YEARS = available_years
    print('YEARS auto-set to all available:', YEARS)
else:
    missing = [y for y in YEARS if y not in available_years]
    assert not missing, f'Years not available with 4 bands at this AOI: {missing}'

## 6. Helper: export NAIP AOI clip with date-tagged filename

In [ ]:
import datetime as _dt
import geemap

LOCAL_NAIP = REPO / 'data' / 'naip_aoi'
LOCAL_CHM = REPO / 'output'
LOCAL_NAIP.mkdir(parents=True, exist_ok=True)
LOCAL_CHM.mkdir(parents=True, exist_ok=True)

def _median_acq_date(year_collection):
    """Return YYYYMMDD string for the median acquisition date of contributing images."""
    dates_ms = year_collection.aggregate_array('acq_ms').getInfo()
    assert dates_ms, 'No images in year collection'
    dates_ms.sort()
    median_ms = dates_ms[len(dates_ms) // 2]
    return _dt.datetime.utcfromtimestamp(median_ms / 1000).strftime('%Y%m%d')

def export_naip_aoi(year):
    """Export a 4-band NAIP AOI clip for one year. Returns the local file path and date tag."""
    yr_col = tagged.filter(ee.Filter.calendarRange(year, year, 'year'))
    n = yr_col.size().getInfo()
    assert n > 0, f'No 4-band NAIP images for {SITE} in {year}'
    date_tag = _median_acq_date(yr_col)

    mosaic = yr_col.select(['R', 'G', 'B', 'N']).mosaic().clip(aoi)
    fname = f'NAIP_{SITE}_{date_tag}_aoi{int(HALF_SIZE_KM*2)}km.tif'
    drive_path = DRIVE_NAIP / fname
    local_path = LOCAL_NAIP / fname

    if drive_path.exists():
        print(f'  Cached: {drive_path.name}')
        if not local_path.exists():
            os.symlink(drive_path, local_path)
        return local_path, date_tag

    print(f'  Downloading {fname} via geemap...')
    geemap.ee_export_image(
        mosaic.toUint8(),
        filename=str(local_path),
        scale=0.6,
        region=aoi,
        crs='EPSG:5070',
        file_per_band=False,
    )
    # Cache to Drive for next time.
    import shutil; shutil.copy2(local_path, drive_path)
    return local_path, date_tag

## 7. Run inference for each year

In [ ]:
import subprocess, time
import torch
print('CUDA available:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

results = []
for year in YEARS:
    print(f'\n=== {SITE} {year} ===')
    t0 = time.time()
    naip_path, date_tag = export_naip_aoi(year)
    print(f'  NAIP AOI ready in {time.time()-t0:.1f}s: {naip_path}')

    chm_out_dir = LOCAL_CHM / f'{SITE}_{date_tag}'
    chm_out_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        'python', 'scripts/inference.py',
        '--naip-quad', str(naip_path),
        '--output-dir', str(chm_out_dir),
        '--model-checkpoint', MODEL_PATH,
        '--config', CONFIG_PATH,
        '--static-rasters-dir', CONDITIONING_DIR,
    ]
    print('  Running inference...')
    t1 = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t1
    if r.returncode != 0:
        print('  STDERR:', r.stderr[-2000:])
        raise RuntimeError(f'Inference failed for {year}')
    chm_files = sorted(chm_out_dir.glob('*_chm.tif'))
    assert chm_files, f'No CHM output in {chm_out_dir}'
    chm_path = chm_files[-1]
    # Copy to Drive for retrieval.
    import shutil
    drive_chm = DRIVE_CHM / chm_path.name
    shutil.copy2(chm_path, drive_chm)
    print(f'  CHM done in {elapsed:.1f}s -> {drive_chm}')
    results.append({'year': year, 'date_tag': date_tag, 'naip': naip_path, 'chm': drive_chm, 'seconds': elapsed})

print('\n=== Summary ===')
for r in results:
    print(f"  {r['year']} ({r['date_tag']}): {r['seconds']:.1f}s -> {r['chm'].name}")

## 8. Quick-look montage

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt

n = len(results)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5), squeeze=False)
for ax, r in zip(axes[0], results):
    with rasterio.open(r['chm']) as src:
        arr = src.read(1).astype(np.float32)
        arr[arr == src.nodata] = np.nan
        arr = arr / 100.0  # cm -> m
    im = ax.imshow(arr, cmap='viridis', vmin=0, vmax=30)
    ax.set_title(f"{SITE} {r['year']} ({r['date_tag']})")
    ax.set_axis_off()
fig.colorbar(im, ax=axes[0].tolist(), shrink=0.7, label='Height (m)')
plt.show()

## 9. Retrieve outputs

Outputs are now in `MyDrive/NAIP_CHM_PoC/chm_predictions/{SITE}/`. On your laptop, copy them into `ProofOfConcept/Data/NAIP/Predicted_CHM/{SITE}/`.